<a href="https://colab.research.google.com/github/KEY-FOX/colab_ex/blob/main/%EC%8B%A4%EC%8A%B53/GRU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3. IMDB: 리뷰를 읽고 긍정·부정 판단하기

이미지는 픽셀이 이미 숫자이지만 리뷰는 단어로 되어 있습니다. 모델에 넣으려면 먼저 단어에 번호를 붙이고, 각 단어를 계산 가능한 벡터로 바꾸어야 합니다.

이 모델은 리뷰를 단어 순서대로 읽습니다. 예를 들어 “good”과 “not good”은 같은 단어가 포함되어도 의미가 다릅니다. GRU는 앞에서 처리한 정보와 현재 단어를 함께 사용해 리뷰를 표현하도록 학습합니다. 순서를 사용한다고 해서 모든 부정 표현이나 문맥을 정확히 이해하는 것은 아닙니다.

**리뷰 → 단어 번호 → 단어 벡터 → GRU → 긍정·부정 점수**

## 3-1. 실행 환경과 학습 설정

IMDB 리뷰는 Hugging Face의 `datasets`로 불러오고, 배치는 PyTorch의 DataLoader로 구성합니다. 첫 설치 셀은 필요한 데이터 라이브러리를 설치합니다. Colab에서는 GPU 런타임을 선택하면 학습 시간을 줄일 수 있습니다.

Embedding은 단어 하나를 128개의 숫자로 표현합니다. GRU는 앞에서 읽은 정보를 256개의 숫자로 표현하며, 3층으로 구성합니다. 긴 리뷰는 앞의 300개 단어까지 사용합니다. 이 제한을 늘리면 더 많은 내용을 읽지만 계산량도 늘어납니다.

설정을 변경하면 데이터 준비부터 다시 실행합니다. 특히 모델을 새로 생성한 뒤에는 optimizer도 다시 생성하고 학습해야 합니다.

In [1]:
%pip install -q datasets

import re
from collections import Counter
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from datasets import load_dataset

In [2]:
# 계산 장치
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)

# 학습 설정
batch_size = 64
num_epochs = 5

out_node = 1         # 긍정·부정을 판단할 점수 하나를 출력
embedd_dim = 128     # 단어 하나를 128개의 숫자로 표현
valid_percent = 0.2  # 학습 데이터의 20%를 검증용으로 분리
MIN_FREQ = 10        # 학습 리뷰에서 10회 이상 등장한 단어를 등록
MAX_LENGTH = 300     # 리뷰마다 앞의 300개 단어까지 사용

num_layers = 3     # TODO: 1, 2, 3을 비교하세요. 기본값: 3
learning_rate = 0.005  # TODO: 0.0001, 0.001, 0.005를 비교하세요. 기본값: 0.005
hidden_dim = 256     # TODO: 64, 128, 256을 비교하세요. 기본값: 256

# 한 번에 하나만 변경하고, 나머지 두 값은 기본값으로 고정하세요.
# 변경 후 모델과 optimizer를 다시 생성하고 학습하세요.
# 검증 정확도와 검증 손실을 비교하여 설정을 선택하세요.

print("계산 장치:", device)

계산 장치: cuda


## 3-2. 리뷰와 정답 불러오기

IMDB에는 학습용 리뷰 25,000개와 테스트용 리뷰 25,000개가 있습니다. 정답은 부정 0, 긍정 1입니다. 정답이 없는 추가 데이터는 이번 실습에서 사용하지 않습니다.

리뷰를 소문자로 바꾸고 HTML 줄바꿈을 공백으로 바꾼 뒤, 공백 기준으로 단어를 나눕니다. 문장부호를 별도로 분석하는 정교한 전처리는 아니며, 이후 모든 리뷰에 같은 규칙을 적용합니다.

In [3]:
def tokenize(text):
    text = re.sub(r"<br\s*/?>", " ", text.lower())
    return text.split()

train_data = load_dataset("stanfordnlp/imdb", split="train")
test_data = load_dataset("stanfordnlp/imdb", split="test")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
print("train:", len(train_data), "test:", len(test_data))
print("리뷰:", train_data[0]["text"])
print("정답:", train_data[0]["label"], "(0: 부정, 1: 긍정)")
print("단어:", tokenize(train_data[0]["text"])[:20])

train: 25000 test: 25000
리뷰: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes

## 3-3. 검증 데이터 분리

학습 리뷰 중 20%는 검증에 사용합니다. 학습 데이터로 가중치를 업데이트하고, 검증 손실로 저장할 모델을 선택합니다. 테스트 리뷰는 최종 평가 때 사용합니다.

긍정·부정 비율을 유지하면서 나누며, 같은 분할을 재현하도록 seed를 지정합니다.

In [5]:
split_data = train_data.train_test_split(
    test_size=valid_percent, seed=42, stratify_by_column="label"
)
train_data = split_data["train"]
valid_data = split_data["test"]

In [6]:
print("train:", len(train_data))
print("valid:", len(valid_data))
print("test :", len(test_data))

train: 20000
valid: 5000
test : 25000


## 3-4. 단어 목록과 단어 번호 만들기

학습 리뷰에 등장한 단어의 횟수를 세어 10회 이상 등장한 단어를 등록합니다. 검증·테스트 리뷰는 단어 목록을 만들 때 사용하지 않습니다.

`<pad>`는 배치의 길이를 맞추는 빈자리이며 번호는 0입니다. `<unk>`는 목록에 없는 단어이며 번호는 1입니다. 나머지 단어에는 2부터 번호를 부여합니다. 단어 번호의 크기는 단어의 의미나 중요도를 나타내지 않습니다.

In [7]:
word_counts = Counter()
for review in train_data["text"]:
    word_counts.update(tokenize(review))

vocab = {"<pad>": 0, "<unk>": 1}
for word, count in word_counts.items():
    if count >= MIN_FREQ:
        vocab[word] = len(vocab)

PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

In [8]:
vocab_size = len(vocab)
print("단어 목록의 크기:", vocab_size)
print("단어 번호 예시:", list(vocab.items())[:10])

단어 목록의 크기: 22833
단어 번호 예시: [('<pad>', 0), ('<unk>', 1), ('after', 2), ('reading', 3), ('tons', 4), ('of', 5), ('good', 6), ('reviews', 7), ('about', 8), ('this', 9)]


## 3-5. 길이가 다른 리뷰를 배치로 구성

리뷰의 단어를 번호로 바꾸고 앞의 300개까지만 사용합니다. 짧은 리뷰 뒤에는 `<pad>`를 채워 배치 안의 길이를 맞춥니다. DataLoader는 단어 번호, 실제 리뷰 길이, 정답을 함께 전달합니다.

입력은 `[리뷰 수, 단어 위치 수]` 형태입니다. 실제 길이도 전달하므로 모델은 padding을 제외한 리뷰의 마지막 상태를 사용할 수 있습니다. 단어 번호는 정수형, 이진 분류 정답은 실수형으로 준비합니다.

In [9]:
class ReviewDataset(Dataset):
    def __init__(self, reviews):
        self.samples = []
        for review in reviews:
            words = tokenize(review["text"])[:MAX_LENGTH]
            ids = [vocab.get(word, UNK_IDX) for word in words]
            if not ids:
                ids = [UNK_IDX]
            self.samples.append((torch.tensor(ids, dtype=torch.long), review["label"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]

def make_batch(samples):
    texts, labels = zip(*samples)
    lengths = torch.tensor([len(text) for text in texts], dtype=torch.long)
    texts = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    labels = torch.tensor(labels, dtype=torch.float32)
    return texts, lengths, labels

train_loader = DataLoader(ReviewDataset(train_data), batch_size=batch_size,
                          shuffle=True, collate_fn=make_batch)
valid_loader = DataLoader(ReviewDataset(valid_data), batch_size=batch_size,
                          shuffle=False, collate_fn=make_batch)
test_loader = DataLoader(ReviewDataset(test_data), batch_size=batch_size,
                         shuffle=False, collate_fn=make_batch)

text, lengths, label = next(iter(train_loader))
print("단어 번호 형태:", text.shape)
print("실제 길이:", lengths[:5].tolist())
print("정답:", label[:5].tolist())

단어 번호 형태: torch.Size([64, 300])
실제 길이: [285, 114, 251, 300, 241]
정답: [0.0, 1.0, 0.0, 0.0, 1.0]


## 3-6. Embedding과 GRU로 리뷰 분류 모델 만들기

Embedding은 단어 번호를 단어 벡터로 바꿉니다. GRU는 이 벡터를 순서대로 읽으며 정보를 갱신합니다. 별도의 cell state 없이 hidden state로 정보를 전달합니다.

`pack_padded_sequence`는 실제 길이를 이용해 padding 부분을 GRU 계산에서 제외합니다. `hidden[-1]`은 가장 위 GRU층에서 각 리뷰를 끝까지 읽은 상태입니다. Dropout과 Linear를 거쳐 리뷰마다 분류 점수 하나를 출력합니다. 이 점수에는 아직 sigmoid를 적용하지 않습니다.

In [10]:
class Sentiment(nn.Module):
    def __init__(self, num_vocab, embedding_dim, hidden_dim, num_layers,
                 out_node, drop_percent=0.2):
        super().__init__()
        self.embed = nn.Embedding(num_vocab, embedding_dim, padding_idx=PAD_IDX)
        self.gru = nn.GRU(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.dropout = nn.Dropout(drop_percent)
        self.fc = nn.Linear(hidden_dim, out_node)

    def forward(self, x, lengths):
        x = self.embed(x)
        # 빈자리를 제외하고 각 리뷰의 실제 단어만 처리합니다.
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True,
                                      enforce_sorted=False)
        _, hidden = self.gru(packed)
        # 가장 위층에서 각 리뷰의 마지막 상태를 가져옵니다.
        out = self.dropout(hidden[-1])
        return self.fc(out)

In [11]:
# 모델을 새로 만들 때 손실 함수와 optimizer도 함께 생성합니다.
model = Sentiment(vocab_size, embedd_dim, hidden_dim, num_layers, out_node).to(device)
criterion = nn.BCEWithLogitsLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
assert optimizer.param_groups[0]["params"][0] is next(model.parameters())

## 3-7. 이진 분류 손실과 optimizer

앞의 셀에서 모델과 함께 손실 함수와 optimizer를 생성했습니다. `BCEWithLogitsLoss`는 sigmoid 계산을 포함하므로 모델의 점수를 그대로 받습니다. 정확도를 구할 때는 sigmoid 결과가 0.5 이상이면 긍정 1로 판단합니다.

모델 생성 셀을 다시 실행하면 가중치가 초기화됩니다. 이 경우 다시 학습한 뒤 평가해야 합니다.

In [12]:
print("모델 구조:", model)
print("optimizer 연결 확인:",
      optimizer.param_groups[0]["params"][0] is next(model.parameters()))

모델 구조: Sentiment(
  (embed): Embedding(22833, 128, padding_idx=0)
  (gru): GRU(128, 256, num_layers=3, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)
optimizer 연결 확인: True


## 3-8. 학습 함수

배치마다 리뷰와 정답을 계산 장치로 옮기고, 예측·손실 계산·기울기 계산·가중치 업데이트를 진행합니다. 리뷰 길이는 padding을 제외하는 데 사용합니다.

모델의 출력 `[리뷰 수, 1]`을 `squeeze(1)`로 `[리뷰 수]`로 바꿔 정답과 맞춥니다. 손실은 리뷰 수를 반영하여 평균을 구하고, 정확도는 맞힌 리뷰의 비율로 구합니다. `model.train()`은 Dropout을 학습 모드로 설정합니다.

In [13]:
def train(model, train_loader, optimizer, criterion):
    epoch_loss, correct = 0.0, 0
    model.train()

    for text, lengths, label in train_loader:
        text, label = text.to(device), label.to(device)
        out = model(text, lengths).squeeze(1)
        loss = criterion(out, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        prediction = (torch.sigmoid(out.detach()) >= 0.5)
        correct += (prediction == label).sum().item()
        epoch_loss += loss.item() * label.size(0)

    count = len(train_loader.dataset)
    return epoch_loss / count, correct / count * 100

### 평가 함수

평가에서는 가중치를 업데이트하지 않습니다. `model.eval()`로 Dropout을 끄고, `torch.no_grad()`로 기울기 계산을 생략합니다. 같은 함수를 검증과 테스트에 사용합니다.

In [14]:
def evaluate(model, data_loader, criterion):
    epoch_loss, correct = 0.0, 0
    model.eval()

    with torch.no_grad():
        for text, lengths, label in data_loader:
            text, label = text.to(device), label.to(device)
            out = model(text, lengths).squeeze(1)
            loss = criterion(out, label)
            prediction = (torch.sigmoid(out) >= 0.5)
            correct += (prediction == label).sum().item()
            epoch_loss += loss.item() * label.size(0)

    count = len(data_loader.dataset)
    return epoch_loss / count, correct / count * 100

### 학습 실행과 모델 저장 및 테스트

각 epoch에서 학습 결과와 검증 결과를 출력합니다. 검증 손실이 가장 낮을 때의 가중치를 저장하고, 학습 종료 후 해당 가중치로 테스트합니다. 테스트 데이터로 저장할 모델을 선택하지 않습니다.

저장 경로는 `gru_checkpoints/GRU_best.pth`입니다. 최종 결과는 마지막 epoch가 아니라 검증 손실로 선택한 모델의 테스트 정확도입니다. 전체 데이터 학습에는 시간이 걸리며, 실제 정확도는 실행 결과로 확인합니다.

In [15]:
checkpoint_dir = Path("gru_checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
checkpoint_path = checkpoint_dir / "GRU_best.pth"

print("____GRU____")
best_valid_loss = float("inf")
best_epoch = 0
for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, optimizer, criterion)
    valid_loss, valid_acc = evaluate(model, valid_loader, criterion)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)

    print(f"Epoch[{epoch+1}/{num_epochs}], Train Loss:{train_loss:.4f}, "
          f"Train Acc:{train_acc:.2f}, Valid Loss:{valid_loss:.4f}, "
          f"Valid Acc:{valid_acc:.2f}")

# 검증 손실이 가장 낮았던 모델로 최종 평가합니다.
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
GRU_loss, GRU_acc = evaluate(model, test_loader, criterion)
print(f"선택한 epoch: {best_epoch}")
print(f"GRU's IMDB Accuracy:{GRU_acc:.2f}% (Loss:{GRU_loss:.4f})")

____GRU____
Epoch[1/5], Train Loss:0.6971, Train Acc:52.84, Valid Loss:0.6946, Valid Acc:51.74
Epoch[2/5], Train Loss:0.7003, Train Acc:53.11, Valid Loss:0.6958, Valid Acc:52.80
Epoch[3/5], Train Loss:0.6841, Train Acc:55.96, Valid Loss:0.6660, Valid Acc:63.76
Epoch[4/5], Train Loss:0.5187, Train Acc:74.06, Valid Loss:0.4346, Valid Acc:79.60
Epoch[5/5], Train Loss:0.3301, Train Acc:86.23, Valid Loss:0.4649, Valid Acc:78.98
선택한 epoch: 4
GRU's IMDB Accuracy:78.44% (Loss:0.4488)
